# 01 · Data download and loader checks

Downloads the two candidate datasets for decision D-004 at pinned commits and checks every function in `src/data.py` on small synthetic inputs.

- Run top to bottom after a kernel restart.
- The unit checks need no network; only the download section does.
- Clear all outputs before committing: outputs can contain dataset text.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")  # repo root, so `src` is importable from notebooks/

import json
import tempfile
import zipfile
from pathlib import Path

from src import data

print(data.ROOT)

## Unit checks (synthetic data, no network)

Each cell exercises one function and fails with an `AssertionError` if the behaviour is wrong. `assert` is fine here: these are tests, not pipeline validation.

In [ ]:
# _check: silent when the condition holds, ValueError with the message when it does not
data._check(1 + 1 == 2, "never shown")
try:
    data._check(len([1, 2]) == 3, "expected 3 items")
    raise AssertionError("_check did not raise")
except ValueError as e:
    assert str(e) == "expected 3 items"
print("_check OK")

In [ ]:
# sha256_of: must reproduce the published SHA-256 of b"abc"
tmp = Path(tempfile.mkdtemp()) / "abc.txt"
tmp.write_bytes(b"abc")
assert data.sha256_of(tmp) == "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"
print("sha256_of OK")

In [ ]:
# _update_manifest: keeps other keys, replaces the same key, writes only to manifest_path
real_before = data.MANIFEST.read_text() if data.MANIFEST.exists() else None
tmp_manifest = Path(tempfile.mkdtemp()) / "MANIFEST.json"
try:
    data._update_manifest("a", {"x": 1}, manifest_path=tmp_manifest)
    data._update_manifest("b", {"y": 2}, manifest_path=tmp_manifest)
    data._update_manifest("a", {"x": 3}, manifest_path=tmp_manifest)
    assert tmp_manifest.exists(), "_update_manifest ignored manifest_path"
    assert json.loads(tmp_manifest.read_text()) == {"a": {"x": 3}, "b": {"y": 2}}
finally:
    # never leave test entries in the real manifest
    if real_before is None:
        data.MANIFEST.unlink(missing_ok=True)
    else:
        data.MANIFEST.write_text(real_before)
print("_update_manifest OK")

In [ ]:
# load_phrasebank: toy zip with the real member names, CRLF line endings,
# a blank line and an "@" inside a sentence
toy_zip = Path(tempfile.mkdtemp()) / "toy.zip"
toy_text = "Sales fell 10 %@negative\r\nContact ir@x.fi for details@neutral\r\n\r\nProfit doubled@positive\r\n"
with zipfile.ZipFile(toy_zip, "w") as zf:
    for member in data.PHRASEBANK_FILES.values():
        zf.writestr(member, toy_text.encode(data.PHRASEBANK_ENCODING))

toy = data.load_phrasebank(toy_zip, expected_rows={k: 3 for k in data.PHRASEBANK_FILES})
assert len(toy) == 12
assert sorted(toy["row_in_file"].unique()) == [0, 1, 3]  # the blank line keeps its position
assert toy.loc[1, "text"] == "Contact ir@x.fi for details"
assert toy.loc[1, "label"] == "neutral"

try:
    data.load_phrasebank(toy_zip, expected_rows={k: 4 for k in data.PHRASEBANK_FILES})
    raise AssertionError("row-count check did not fire")
except ValueError as e:
    print("row-count check fires:", e)
print("load_phrasebank OK")

In [ ]:
# load_twitter: quoted newline stays inside one text; an unknown label is rejected
tmpdir = Path(tempfile.mkdtemp())
(tmpdir / "a.csv").write_text('text,label\n"Shares slide 2%",0\n"Line one\nline two",2\n')
(tmpdir / "b.csv").write_text('text,label\n"Profit up",1\n"Odd one",3\n')

ok = data.load_twitter({"a.csv": tmpdir / "a.csv"}, expected_rows={"a.csv": 2})
assert len(ok) == 2
assert ok.loc[1, "text"] == "Line one\nline two"
assert list(ok["label_name"]) == ["bearish", "neutral"]
assert list(ok["row_in_file"]) == [0, 1]

try:
    data.load_twitter({"b.csv": tmpdir / "b.csv"}, expected_rows={})
    raise AssertionError("unknown-label check did not fire")
except ValueError as e:
    print("label check fires:", e)
print("load_twitter OK")

## Download (network)

Downloads at the commits pinned in `src/data.py` and records each file in `data/MANIFEST.json`. Re-running is safe: same commit, same files, and each manifest entry is replaced, not duplicated.

In [ ]:
pb_zip = data.download_phrasebank()
tw_paths = data.download_twitter()

print(pb_zip)
for name, path in tw_paths.items():
    print(name, "->", path)
print(data.MANIFEST.read_text())